# Association Rule Mining Pipeline

This notebook runs the full market basket analysis workflow using the Azure PostgreSQL retail warehouse as the source of truth.

It prepares basket-item data, runs Apriori, FP-Growth, and ECLAT, records runtime and memory usage, and exports the final rule files plus an algorithm comparison table.

## What You Will Produce

Expected outputs are written to `outputs/`:

- `basket_items_category.csv`
- `association_rules_apriori.csv`
- `association_rules_fpgrowth.csv`
- `association_rules_eclat.csv`
- `association_rules_comparison.csv`

Generated CSVs are intentionally ignored by Git because they can be large.

## 1. Setup

Install dependencies first if needed:

```bash
pip install -r requirements.txt
```

Keep your real Azure PostgreSQL credentials in a local `.env` file. Do not commit `.env`.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

required_modules = {
    "pandas": "pandas",
    "dotenv": "python-dotenv",
    "sqlalchemy": "sqlalchemy",
    "psycopg2": "psycopg2-binary",
    "psutil": "psutil",
}
missing_packages = [
    package_name
    for module_name, package_name in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages into this notebook Python:")
    print(sys.executable)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(PROJECT_ROOT / "requirements.txt"),
    ])
else:
    print("Python dependencies are available in this notebook kernel.")

import pandas as pd
from dotenv import load_dotenv

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook Python: {sys.executable}")
print(f"Outputs folder: {OUTPUT_DIR}")

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

required_env_vars = [
    "Azure_DB_HOST",
    "Azure_DB_USER",
    "Azure_DB_PASSWORD",
]

missing = [name for name in required_env_vars if not os.getenv(name)]
if missing:
    print("Missing database settings:", missing)
    print("Add them to a local .env file before running the extraction step.")
else:
    print("Database environment looks ready. Credentials were loaded without printing secrets.")

## 2. Choose Mining Level

Start with `product_category`. It is the safest level because the source has many product IDs. After the category run works, try `product_type`. Use `product_id` only if runtime and memory are acceptable.

In [ ]:
ITEM_LEVEL = "product_category"  # options: product_category, product_type, product_id

MIN_SUPPORTS = [0.001, 0.003, 0.005, 0.01]
MIN_CONFIDENCES = [0.3, 0.4, 0.5, 0.6]

# Keep this at 3 for a practical first run. Larger values can grow very quickly.
MAX_ITEMSET_SIZE = 3

# Set to a small number such as 5000 for a fast classroom/demo run.
# Keep as None for the full Azure PostgreSQL dataset.
LIMIT_BASKETS = None

LEVEL_SUFFIX = {
    "product_category": "category",
    "product_type": "type",
    "product_id": "product_id",
}[ITEM_LEVEL]

BASKET_ITEMS_CSV = OUTPUT_DIR / f"basket_items_{LEVEL_SUFFIX}.csv"
print(f"Selected item level: {ITEM_LEVEL}")
print(f"Prepared data will be saved to: {BASKET_ITEMS_CSV}")

## 3. Prepare Basket-Item Data

This step connects to Azure PostgreSQL and reads distinct `basket_id` plus the selected item field from `retail_dw.stg_basket_items_for_mba`.

The script removes nulls, drops duplicate basket-item pairs, saves the CSV, and reports the transaction-list shape used for mining.

In [ ]:
def run_command(command):
    print("Running command:")
    print(" ".join(str(part) for part in command))
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")


prepare_command = [
    sys.executable,
    str(SCRIPTS_DIR / "04_prepare_basket_matrix.py"),
    "--item-level",
    ITEM_LEVEL,
    "--output",
    str(BASKET_ITEMS_CSV),
]

if LIMIT_BASKETS is not None:
    prepare_command.extend(["--limit-baskets", str(LIMIT_BASKETS)])

run_command(prepare_command)

In [ ]:
basket_items = pd.read_csv(BASKET_ITEMS_CSV)
display(basket_items.head(10))

basket_summary = pd.Series(
    {
        "basket_item_rows": len(basket_items),
        "unique_baskets": basket_items["basket_id"].nunique(),
        "unique_items": basket_items["item_name"].nunique(),
        "duplicate_basket_item_pairs": basket_items.duplicated(["basket_id", "item_name"]).sum(),
        "null_basket_id": basket_items["basket_id"].isna().sum(),
        "null_item_name": basket_items["item_name"].isna().sum(),
    }
).to_frame("value")

display(basket_summary)

## 4. Run Apriori, FP-Growth, and ECLAT

The mining script uses the same support and confidence thresholds for each algorithm. It also records runtime and memory usage for every threshold combination.

ECLAT is implemented with a vertical representation: `item_name -> set(basket_id)`.

In [ ]:
rules_command = [
    sys.executable,
    str(SCRIPTS_DIR / "05_association_rules.py"),
    "--item-level",
    ITEM_LEVEL,
    "--input",
    str(BASKET_ITEMS_CSV),
    "--min-supports",
    ",".join(str(value) for value in MIN_SUPPORTS),
    "--min-confidences",
    ",".join(str(value) for value in MIN_CONFIDENCES),
    "--max-itemset-size",
    str(MAX_ITEMSET_SIZE),
    "--output-dir",
    str(OUTPUT_DIR),
]

run_command(rules_command)

## 5. Compare Algorithm Performance

The comparison table is the main place to evaluate speed, memory, number of frequent itemsets, and number of rules under each threshold setting.

In [ ]:
comparison_path = OUTPUT_DIR / "association_rules_comparison.csv"
comparison = pd.read_csv(comparison_path)

display(
    comparison.sort_values(
        ["min_support", "min_confidence", "algorithm_name"],
        ascending=[True, True, True],
    )
)

performance_view = comparison[
    [
        "algorithm_name",
        "min_support",
        "min_confidence",
        "frequent_itemset_count",
        "rule_count",
        "runtime_seconds",
        "memory_usage_mb",
        "memory_delta_mb",
    ]
]

display(performance_view)

## 6. Inspect Strong Rules

Useful interpretation guide:

- `support`: share of baskets containing both sides of the rule.
- `confidence`: probability of the consequent given the antecedent.
- `lift`: relationship strength compared with random co-occurrence. Values above 1 suggest a positive association.

In [ ]:
rule_files = {
    "Apriori": OUTPUT_DIR / "association_rules_apriori.csv",
    "FP-Growth": OUTPUT_DIR / "association_rules_fpgrowth.csv",
    "ECLAT": OUTPUT_DIR / "association_rules_eclat.csv",
}

columns_to_show = [
    "algorithm_name",
    "min_support",
    "min_confidence",
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
]

for algorithm_name, path in rule_files.items():
    rules = pd.read_csv(path)
    print(f"\n{algorithm_name}: {len(rules):,} exported rules")
    if rules.empty:
        print("No rules met the selected thresholds.")
        continue
    display(
        rules.sort_values(["lift", "confidence", "support"], ascending=False)
        .head(10)[columns_to_show]
    )

## 7. Recommended Next Runs

After the category-level run is stable:

1. Change `ITEM_LEVEL` to `product_type` and rerun all cells.
2. Review memory and runtime in `association_rules_comparison.csv`.
3. Try `product_id` only after raising `min_support` or using `LIMIT_BASKETS` for a small test.

For product-level mining, start conservatively because there are many unique products.